# recursive_opt — meta-optimization on Trace / Trace-Bench (demo)

**Goal.** *Recursive (meta) optimization* uses Trace not only to optimize a task
artifact, but to optimize **how that optimization is done** — and then to
optimize *that*, across families of problems:

| Level | Optimizes | Example |
|---|---|---|
| **O0** | a task artifact | a solver prompt / code |
| **O1** | how O0 is optimized | starting artifact, batch size/design, trace type, memory, optimizer+tools, guide, trainer |
| **O2** | the O1 policy per problem family | which setup per family |
| **O3** | transferable priors across families | a default that works on unseen families |

The whole system rests on one idea: **a recursion level is itself a
`trace.Module`**, so the same `opto.trainer.train` + `opto.optimizers` optimizer
drives every level. Two trainable *surfaces*:

* **selection/config** (`LevelConfig` + `MetaLevel`) — pick & configure existing components (example A);
* **code/implementation** (`ComponentSpec` + `CodeArtifactLevel`) — rewrite a component's *source code* to improve or invent components (example B).

This notebook runs four demos **offline** (no API key / GPU). A final section
shows how to switch to the **real LLM optimizer**.


## 0. Setup (Colab or local)
Clones OpenTrace @ PR #73 (graph adapter + OTEL/Sysmon + trainers), installs
OpenTelemetry, and makes the `opto.features.recursive_opt` package + `examples/`
importable. If you run from your own fork where `recursive_opt` is already under
`opto/features/`, the copy step is a no-op.


In [1]:
import os, sys, subprocess, shutil, pathlib, importlib.util
IN_COLAB = 'google.colab' in sys.modules

cwd = pathlib.Path.cwd()
if (cwd/'opto'/'features'/'recursive_opt').exists():
    REPO = cwd
elif (cwd.parent/'opto'/'features'/'recursive_opt').exists():
    REPO = cwd.parent
else:
    REPO = cwd/'OpenTrace'
    if not REPO.exists():
        subprocess.run(['git','clone','--quiet','https://github.com/AgentOpt/OpenTrace.git', str(REPO)], check=True)
        subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin','pull/73/head:pr73'], check=True)
        subprocess.run(['git','-C',str(REPO),'checkout','--quiet','pr73'], check=True)

def ensure_packages(specs):
    missing = [pkg for module, pkg in specs if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable,'-m','pip','install','-q',*missing], check=True)

ensure_packages([
    ('opentelemetry', 'opentelemetry-api'),
    ('opentelemetry.sdk', 'opentelemetry-sdk'),
])
# Optional (real Trace-Bench tasks): pip install -e Trace-Bench'[hf]'  — omitted for the offline demo.

# Ensure the recursive_opt deliverable package sits under opto/features/.
target = pathlib.Path(REPO)/'opto'/'features'/'recursive_opt'
if not target.exists():
    src = pathlib.Path('recursive_opt')  # the deliverable folder next to this notebook
    if src.exists():
        shutil.copytree(src, target)
    else:
        raise FileNotFoundError(f'Place recursive_opt/ under {target} or next to this notebook.')

os.chdir(REPO)
sys.path.insert(0, str(pathlib.Path(REPO).resolve()))
sys.path.insert(0, str((pathlib.Path(REPO)/'examples').resolve()))
import opto.features.recursive_opt as R
print('recursive_opt ready. PR#73 traces:', R.traces.HAVE_PR73, '| Trace-Bench:', R.tracebench.HAVE_TB)


recursive_opt ready. PR#73 traces: False | Trace-Bench: False



## A. Learn the best Trace *setup* (selection/config surface)
An O1 `MetaLevel` optimizes a compact configuration over existing optimization components.
This cell compares the initial weak setup against candidate optimized setups and prints the
selected config for each problem.

**Current capability.** The recursive layer can treat optimizer/trainer/batch/memory choices as a trainable artifact, score each setup through an inner run, record typed memory episodes, and promote per-family priors.

**Current limits.** This surface selects/configures known components; it does not rewrite their implementation. In stub mode the scores are deterministic analytic proxies. Real problem scores require Trace-Bench installed and compatible task adapters.


In [2]:
from examples.recursive_opt_example_A_learn_setup import CANDIDATES, PROBLEMS, build_level
from opto.features.recursive_opt import MemoryLite, RecursiveGuide


def evaluate_setup_candidates(problem):
    mem = MemoryLite(root=f"./mem_A_{problem.split(':')[-1]}")
    level = build_level(problem, mem)
    guide = RecursiveGuide()
    rows = []
    for cand in CANDIDATES:
        level.propose(**cand)
        out = level.forward(problem)
        score, feedback = guide(problem, out, None)
        rows.append({"config": dict(cand), "score": score, "feedback": feedback})
    return rows, mem


A_RESULTS = {}
for problem in PROBLEMS:
    rows, mem = evaluate_setup_candidates(problem)
    initial = next(row for row in rows if row["config"]["batch_design"] == "random")
    best = max(rows, key=lambda row: row["score"])
    A_RESULTS[problem] = {"initial": initial, "optimized": best, "all": rows, "memory": mem.summary()}

    print(f"\n=== A: {problem} ===")
    print("candidate setup scores:")
    for row in rows:
        cfg = row["config"]
        print(
            f"  score={row['score']:.3f} | bs={cfg['batch_size']:<2} "
            f"batch={cfg['batch_design']:<16} memory={cfg['memory_policy']:<9} trainer={cfg['trainer']}"
        )
    print("initial config:", initial["config"], f"score={initial['score']:.3f}")
    print("optimized config:", best["config"], f"score={best['score']:.3f}")
    print("improvement:", f"{best['score'] - initial['score']:+.3f}")
    print("promoted memory priors:", mem.summary()["priors"])



=== A: llm4ad:online_bin_packing_local ===
candidate setup scores:
  score=0.820 | bs=4  batch=failure_balanced memory=typed     trainer=BeamsearchAlgorithm
  score=0.837 | bs=8  batch=curriculum       memory=retrieval trainer=UCBSearchAlgorithm
  score=0.509 | bs=1  batch=random           memory=none      trainer=MinibatchAlgorithm
initial config: {'batch_size': 1, 'batch_design': 'random', 'memory_policy': 'none', 'trainer': 'MinibatchAlgorithm'} score=0.509
optimized config: {'batch_size': 8, 'batch_design': 'curriculum', 'memory_policy': 'retrieval', 'trainer': 'UCBSearchAlgorithm'} score=0.837
improvement: +0.327
promoted memory priors: {'llm4ad:online_bin_packing_local': 0.8369000000000001}

=== A: llm4ad:circle_packing ===
candidate setup scores:
  score=0.820 | bs=4  batch=failure_balanced memory=typed     trainer=BeamsearchAlgorithm
  score=0.837 | bs=8  batch=curriculum       memory=retrieval trainer=UCBSearchAlgorithm
  score=0.509 | bs=1  batch=random           memory=none


## B. Improve a component's **code** (code/implementation surface)
`CodeArtifactLevel` makes a function's source code the trainable parameter. This cell shows the baseline component code, the improved component code, and the score delta.

**Current capability.** The recursive layer can optimize implementation artifacts, not only configuration choices. The same pattern can target batch samplers, trace summarizers, trainer hot paths, or future components if there is an importable baseline function and an evaluator.

**Current limits.** Offline mode installs a hand-written improved implementation to prove the score is climbable; live mode lets the LLM optimizer rewrite the bundle source. The evaluator must be strong enough to catch invalid, unsafe, or overfit code. Production use needs sandboxing and tests around generated code.


In [3]:
import inspect

from opto import trace
from opto.features.recursive_opt import CodeArtifactLevel, ComponentSpec, MemoryLite, RecursiveGuide
from opto.features.recursive_opt.tracebench import make_code_evaluator
from examples.recursive_opt_example_B_improve_component import (
    batch_design_baseline,
    batch_design_improved,
    trace_summarizer_baseline,
    trace_summarizer_improved,
)


def evaluate_component_code(problem, name, baseline, improved, objective):
    spec = ComponentSpec(name=name, baseline=baseline, evaluate=make_code_evaluator(problem, name), objective=objective)
    level = CodeArtifactLevel(spec, memory=MemoryLite(root=f"./mem_B_{name}"))
    guide = RecursiveGuide()

    baseline_out = level.forward(problem)
    baseline_score, baseline_feedback = guide(problem, baseline_out, None)
    baseline_code = inspect.getsource(baseline).strip()

    level._impl = trace.bundle(trainable=True)(improved)
    improved_out = level.forward(problem)
    improved_score, improved_feedback = guide(problem, improved_out, None)
    improved_code = inspect.getsource(improved).strip()

    return {
        "baseline_score": baseline_score,
        "baseline_feedback": baseline_feedback,
        "baseline_code": baseline_code,
        "optimized_score": improved_score,
        "optimized_feedback": improved_feedback,
        "optimized_code": improved_code,
    }


B_RESULTS = {
    "batch_design": evaluate_component_code(
        "llm4ad:online_bin_packing_local",
        "batch_design",
        batch_design_baseline,
        batch_design_improved,
        "maximize held-out pass rate by sampling failing/hard items; keep batch diverse",
    ),
    "trace_summarizer": evaluate_component_code(
        "hf:BBEH",
        "trace_summarizer",
        trace_summarizer_baseline,
        trace_summarizer_improved,
        "preserve error evidence, drop INFO/DEBUG, stay concise",
    ),
}

for name, result in B_RESULTS.items():
    print(f"\n=== B: {name} ===")
    print(f"baseline score : {result['baseline_score']:.3f}")
    print(f"optimized score: {result['optimized_score']:.3f}")
    print(f"delta          : {result['optimized_score'] - result['baseline_score']:+.3f}")
    print("\ninitial code:")
    print(result["baseline_code"])
    print("\noptimized code:")
    print(result["optimized_code"])
    print("\noptimized feedback:", result["optimized_feedback"])



=== B: batch_design ===
baseline score : 0.800
optimized score: 1.000
delta          : +0.200

initial code:
def batch_design_baseline(self, n, k):
    """Pick which task indices go in a training batch. BASELINE = first k.

    A good rewrite should oversample HARD/FAILING items and keep the batch
    diverse, instead of blindly returning range(k)."""
    return list(range(k))

optimized code:
def batch_design_improved(self, n, k):
    """Oversample hard items (here: indices divisible by 3) then fill diversely."""
    hard = [i for i in range(n) if i % 3 == 0]
    rest = [i for i in range(n) if i % 3 != 0]
    picked = (hard + rest)[:k]
    return picked

optimized feedback: [batch_design@llm4ad:online_bin_packing_local] picked [0, 3, 6, 9]; hard_items=4/4; diversity=1.00. good: targets failing items.

=== B: trace_summarizer ===
baseline score : 0.823
optimized score: 0.959
delta          : +0.137

initial code:
def trace_summarizer_baseline(self, trace_text):
    """Compress an epis


## C. Learn a **new capability** from a spec + multiple objectives
A capability artifact is a trainable text policy. This cell scores candidate artifacts against accuracy and cost, then selects the Pareto-aware best artifact.

**Current capability.** The recursive layer can synthesize/select a reusable capability from a natural-language spec, track multiple objectives, and use Pareto/weighted selection rather than a single hidden scalar. Memory, agentic optimizer tools, and HITL gates are available as composable wrappers.

**Current limits.** The offline evaluator inspects the artifact text as a deterministic proxy. A real capability run must connect the artifact to task execution and objective measurement. Multi-objective behavior is only as reliable as the metric definitions and validation set.


In [4]:
from opto.trainer.objectives import ObjectiveConfig, pareto_rank, select_best
from opto.features.recursive_opt.tracebench import make_multiobjective_evaluator
from examples.recursive_opt_example_C_learn_capability import (
    CANDIDATE_IMPLS,
    CapabilityArtifact,
    OBJECTIVES,
    PROBLEMS,
)


evaluator = make_multiobjective_evaluator(PROBLEMS, OBJECTIVES)
obj_cfg = ObjectiveConfig(
    mode="pareto",
    minimize={"cost"},
    weights={"accuracy": 1.0, "cost": 1.0},
    tie_break="weighted",
)

scored = []
for impl in CANDIDATE_IMPLS:
    artifact = CapabilityArtifact(seed_impl=impl, evaluator=evaluator)
    aggregate = {"accuracy": 0.0, "cost": 0.0}
    for problem in PROBLEMS:
        objectives = artifact.forward(problem)["objectives"]
        for metric in aggregate:
            aggregate[metric] += objectives[metric] / len(PROBLEMS)
    scored.append((aggregate, impl))

normalized = [{"accuracy": s["accuracy"], "cost": -s["cost"]} for s, _ in scored]
ranks = pareto_rank(normalized, metrics=("accuracy", "cost"))
best_idx = select_best(scored, obj_cfg)
initial_score, initial_impl = scored[0]
best_score, best_impl = scored[best_idx]

print("candidate capability artifacts:")
for idx, ((score, impl), rank) in enumerate(zip(scored, ranks)):
    marker = "<-- selected" if idx == best_idx else ""
    print(f"  rank={rank} acc={score['accuracy']:.2f} cost={score['cost']:.2f} {marker}\n    {impl}")

print("\ninitial artifact:", initial_impl)
print(f"initial objectives: accuracy={initial_score['accuracy']:.2f} cost={initial_score['cost']:.2f}")
print("\noptimized artifact:", best_impl)
print(f"optimized objectives: accuracy={best_score['accuracy']:.2f} cost={best_score['cost']:.2f}")
print(
    "objective delta:",
    f"accuracy={best_score['accuracy'] - initial_score['accuracy']:+.2f}",
    f"cost={best_score['cost'] - initial_score['cost']:+.2f}",
)


candidate capability artifacts:
  rank=0 acc=0.45 cost=0.31 
    Answer directly.
  rank=0 acc=0.65 cost=0.33 
    Make a short plan, then answer.
  rank=0 acc=0.95 cost=0.40 <-- selected
    Make a short plan; execute; then VERIFY/CHECK the answer against the question before responding. Keep it terse.
  rank=1 acc=0.45 cost=0.40 
    Write an extremely detailed multi-paragraph chain-of-thought that re-derives and verifies everything at length.

initial artifact: Answer directly.
initial objectives: accuracy=0.45 cost=0.31

optimized artifact: Make a short plan; execute; then VERIFY/CHECK the answer against the question before responding. Keep it terse.
optimized objectives: accuracy=0.95 cost=0.40
objective delta: accuracy=+0.50 cost=+0.09



## D. Learn A/B/C across families → induce a cross-family prior
This cell runs the setup search across two families, shows the initial weak setup versus the best setup per family, then promotes choices that win across families.

**Current capability.** The recursive stack can compare O1 choices across task families and induce simple O3 transferable priors, such as a default batch design, memory policy, trainer, and trace type that repeatedly win.

**Current limits.** The current O3 prior is a vote over winning config fields, not a learned policy model. It does not yet prove transfer to unseen real tasks unless Trace-Bench is installed and a held-out family evaluation is run. Empty or conflicting priors are valid outcomes when families need different setups.


In [5]:
from opto.features.recursive_opt import LevelConfig, MetaLevel, RecursiveGuide
from opto.features.recursive_opt.tracebench import make_inner_runner
from examples.recursive_opt_example_D_cross_family import FAMILIES, SEARCH, induce_cross_family_prior


def score_family_config(tasks, config):
    guide = RecursiveGuide()
    scores = []
    for task in tasks:
        level = MetaLevel(cfg=LevelConfig(**config), inner_runner=make_inner_runner(task), trainable_fields=tuple(config.keys()))
        score, _ = guide(task, level.forward(task), None)
        scores.append(score)
    return sum(scores) / len(scores)


D_RESULTS = {}
per_family_best = {}
for family, tasks in FAMILIES.items():
    rows = []
    for config in SEARCH:
        rows.append({"config": dict(config), "score": score_family_config(tasks, config)})
    initial = next(row for row in rows if row["config"]["batch_design"] == "random")
    best = max(rows, key=lambda row: row["score"])
    D_RESULTS[family] = {"initial": initial, "optimized": best, "all": rows}
    per_family_best[family] = (best["score"], best["config"])

    print(f"\n=== D: family {family} ===")
    print("tasks:", tasks)
    for row in rows:
        cfg = row["config"]
        print(
            f"  score={row['score']:.3f} | batch={cfg['batch_design']:<16} "
            f"memory={cfg['memory_policy']:<9} trainer={cfg['trainer']:<20} trace={cfg['trace_type']}"
        )
    print("initial setup:", initial["config"], f"score={initial['score']:.3f}")
    print("optimized setup:", best["config"], f"score={best['score']:.3f}")
    print("improvement:", f"{best['score'] - initial['score']:+.3f}")

prior = induce_cross_family_prior(per_family_best)
print("\nO3 induced cross-family prior:", prior if prior else "<none>")



=== D: family combinatorial ===
tasks: ['llm4ad:online_bin_packing_local', 'llm4ad:circle_packing']
  score=0.878 | batch=failure_balanced memory=typed     trainer=BeamsearchAlgorithm  trace=hybrid
  score=0.859 | batch=curriculum       memory=retrieval trainer=UCBSearchAlgorithm   trace=otel
  score=0.593 | batch=random           memory=none      trainer=MinibatchAlgorithm   trace=internal
initial setup: {'batch_design': 'random', 'memory_policy': 'none', 'trainer': 'MinibatchAlgorithm', 'trace_type': 'internal'} score=0.593
optimized setup: {'batch_design': 'failure_balanced', 'memory_policy': 'typed', 'trainer': 'BeamsearchAlgorithm', 'trace_type': 'hybrid'} score=0.878
improvement: +0.285

=== D: family qa_reasoning ===
tasks: ['hf:GSM8K', 'internal:multiobjective_bbeh']
  score=0.878 | batch=failure_balanced memory=typed     trainer=BeamsearchAlgorithm  trace=hybrid
  score=0.859 | batch=curriculum       memory=retrieval trainer=UCBSearchAlgorithm   trace=otel
  score=0.593 | bat


## Live mode (real LLM optimizer)
Offline cells above use deterministic candidate lists or hand-written improved artifacts so the mechanics are inspectable without credentials.

**What live mode really does.** In live mode, the trainable node is passed to Trace's real optimizer (`OptoPrime` / `OptoPrimeMulti`). The model reads the traced execution, scalar/vector score, and feedback, then proposes a new value for the trainable config/code/artifact. The trainer evaluates that proposal and keeps or rejects it according to the search algorithm.

**What it does not automatically mean.** Live mode does not automatically install Trace-Bench, run real benchmark tasks, or guarantee a better proposal. In this notebook, live mode still uses the offline `tracebench.py` stub unless Trace-Bench is installed. The bounded settings below use one beam, one proposal, and one depth so the notebook is cheap to run; remove those environment overrides for a larger search.

**Default model.** The notebook now defaults `TRACE_LITELLM_MODEL` to `gpt-5.4-nano`. If your API account or LiteLLM does not recognize that model, set `TRACE_LITELLM_MODEL` to an accessible model before running the cell.


In [6]:
import getpass, os, re, runpy, sys

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')  # not echoed

os.environ.setdefault('TRACE_LITELLM_MODEL', 'gpt-5.4-nano')

# Keep the live notebook demo bounded; remove these for a larger search.
os.environ.setdefault('RECURSIVE_OPT_LIVE_BEAM_WIDTH', '1')
os.environ.setdefault('RECURSIVE_OPT_LIVE_NUM_PROPOSALS', '1')
os.environ.setdefault('RECURSIVE_OPT_LIVE_MAX_DEPTH', '1')
os.environ.setdefault('RECURSIVE_OPT_LIVE_VALIDATION_SIZE', '1')
os.environ.setdefault('RECURSIVE_OPT_LIVE_BATCH_SIZE', '1')
os.environ.setdefault('RECURSIVE_OPT_LIVE_NUM_EPOCHS', '1')
os.environ.setdefault('RECURSIVE_OPT_LIVE_NUM_THREADS', '1')


def redact_error(text):
    text = re.sub(r"sk-[A-Za-z0-9_-]+", "sk-***", str(text))
    text = re.sub(r"proj_[A-Za-z0-9]+", "proj_***", text)
    return text


def can_access_openai_model(model):
    try:
        from openai import OpenAI
        OpenAI(api_key=os.environ['OPENAI_API_KEY']).models.retrieve(model)
        return True, ""
    except Exception as exc:
        return False, redact_error(exc)


model = os.environ['TRACE_LITELLM_MODEL']
print('Live optimizer model:', model)
print('Live budget: beam_width=1, num_proposals=1, max_depth=1, batch_size=1')
print('Running Example A live: the LLM proposes config values; the trainer validates them.\n')

accessible, access_error = can_access_openai_model(model)
if not accessible:
    print('Live run skipped before optimizer launch.')
    print(f'model={model} access_error={access_error}')
    print('Set TRACE_LITELLM_MODEL to a model your account can access, then rerun this cell.')
else:
    old_argv = sys.argv[:]
    try:
        sys.argv = ['recursive_opt_example_A_learn_setup.py', '--live']
        runpy.run_path('examples/recursive_opt_example_A_learn_setup.py', run_name='__main__')
    except Exception as exc:
        print('\nLive run failed.')
        print(f"model={model} error={type(exc).__name__}: {redact_error(exc)}")
        print('Set TRACE_LITELLM_MODEL to a model your account can access, then rerun this cell.')
    finally:
        sys.argv = old_argv


Live optimizer model: gpt-5.4-nano
Live budget: beam_width=1, num_proposals=1, max_depth=1, batch_size=1
Running Example A live: the LLM proposes config values; the trainer validates them.



Live run skipped before optimizer launch.
model=gpt-5.4-nano access_error=Error code: 404 - {'error': {'message': 'That model does not exist', 'type': 'invalid_request_error', 'param': 'id', 'code': None}}
Set TRACE_LITELLM_MODEL to a model your account can access, then rerun this cell.
